# Sprint 4 — Data Science and Statistical Computing
**Transcrição → Processamento → Modelo → Indicadores → Recomendação**

Este notebook roda de ponta a ponta no Google Colab (Ambiente de execução > Executar tudo).
Atenção: os dados de treino são **sintéticos** (o repositório não tinha as transcrições reais);
para usar dados reais, substitua `data/transcricoes_treino.csv` (colunas `id, transcricao, produto, sentimento, churn, oportunidade`).

## 1. Preparação (clona o repositório e instala dependências)

In [ ]:
import os
if not os.path.exists("sprint4Nemec"):
    !git clone -q https://github.com/Guiroselli/sprint4Nemec.git
%cd sprint4Nemec
!pip -q install scikit-learn pandas numpy joblib

## 2. Dados e processamento

In [ ]:
import pandas as pd
from src import config
from src.processing import limpar, texto_processado, termos_frequentes

df = pd.read_csv(config.TREINO_CSV)
print(df.shape)
display(df.head(3))
print(df.produto.value_counts(), "\n")
print(df.sentimento.value_counts())

ex = df.transcricao[0]
print("\n--- ORIGINAL ---\n", ex)
print("\n--- PROCESSADO ---\n", texto_processado(ex))

## 3. Modelo: comparação e seleção
Quatro classificadores (Regressão Logística, Naive Bayes, LinearSVC calibrado, Random Forest) sobre TF-IDF,
para cada tarefa (produto, sentimento, churn, oportunidade). Critério: F1 macro em validação cruzada de 5 folds.

In [ ]:
!python -m src.train
comp = pd.read_csv("reports/comparacao_modelos.csv")
display(comp)

In [ ]:
print(open("reports/modelo_selecionado.md", encoding="utf-8").read())

## 4. Indicadores e recomendação para uma nova transcrição

In [ ]:
from src.pipeline import analisar, formatar

transcricao = """
Atendente: Bom dia, em que posso ajudar?
Cliente: Estou muito irritado, o cartão foi cobrado duas vezes na fatura e ninguém resolve. Já liguei três vezes.
Atendente: Peço desculpas, vou abrir um protocolo.
Cliente: Se não estornarem, eu cancelo e vou para a concorrência.
"""
print(formatar(analisar(transcricao)))

### Teste com sua própria transcrição
Edite o texto da caixa e rode a célula.

In [ ]:
#@title Cole a transcrição aqui { display-mode: "form" }
texto = "Atendente: Ol\u00e1!\nCliente: Quero investir mais, o que voc\u00eas t\u00eam com melhor rentabilidade? Tenho interesse em previd\u00eancia." #@param {type:"string"}
print(formatar(analisar(texto.replace("\\n", "\n"))))

## 5. Validação com novas transcrições
- **Sintéticas**: seed diferente do treino (otimista, teste de regressão).
- **Manuais**: escritas à mão; as regras foram calibradas olhando estes casos (não independente).
- **Cegas**: escritas à mão, nunca usadas para ajuste → melhor estimativa de generalização (n pequeno).

In [ ]:
!python -m src.validate
print(open("reports/validacao.md", encoding="utf-8").read())

In [ ]:
d = pd.read_csv("reports/validacao_detalhe_cegas.csv")
display(d[["produto", "pred_produto", "sentimento", "pred_sentimento", "churn", "pred_churn",
           "oportunidade", "pred_oportunidade", "risco_churn", "score_oportunidade", "prioridade"]])

## 6. Painel em lote (todas as transcrições novas)

In [ ]:
novas = pd.read_csv(config.NOVAS_CEGAS_CSV)
res = pd.DataFrame([analisar(t) for t in novas.transcricao])
painel = pd.concat([novas[["id"]], res[["produto", "sentimento", "risco_churn", "oportunidade", "prioridade", "recomendacao"]]], axis=1)
display(painel)
print(painel.prioridade.value_counts())
painel.prioridade.value_counts().plot(kind="bar", title="Transcrições por prioridade");

## 7. Reuniões reais do Proton Flow (transcrições anonimizadas + catálogo TOTVS)
Usa as 30 reuniões reais do projeto `TOTUVsVers-oFinal`. O produto é identificado pelo **catálogo TOTVS** (palavras fortes por
produto) e os indicadores usam sinais léxicos com contexto (ex.: "cancelamento de nota" não conta como churn).
Não há gabarito humano: veja `reports/revisao_humana.csv` para validar de verdade.

In [ ]:
if not os.path.exists("../totvs_proton"):
    !git clone -q https://github.com/Guiroselli/TOTUVsVers-oFinal.git ../totvs_proton
!python -m src.importar_totvs ../totvs_proton/backend/dataset_limpo.json
!python -m src.validate_totvs

In [ ]:
from src.totvs_pipeline import analisar_reuniao, carregar_reunioes, formatar
reunioes = {r["id"]: r for r in carregar_reunioes()}
print(formatar(analisar_reuniao(reunioes["1057714"]["transcricao"])))

In [ ]:
real = pd.read_csv("reports/analise_reunioes_reais.csv")
display(real[["id", "cliente", "produto", "sentimento", "risco_churn", "oportunidade", "prioridade", "termos"]])
real.produto.value_counts().plot(kind="bar", title="Produto TOTVS dominante por reunião");

## 8. Recomendações de negócio, limitações e próximos passos
Veja o `README.md` do repositório para o detalhamento completo:
https://github.com/Guiroselli/sprint4Nemec#readme

**Resumo:** retenção em até 24h para churn ≥ 65; não vender para cliente insatisfeito (resolver primeiro);
fila de leads para oportunidade ≥ 50; monitorar termos de churn.
**Limitações:** dados sintéticos, validação cega com apenas 8 casos, sentimento é a tarefa mais frágil (75% no cego).
**Próximos passos:** treinar com transcrições reais, ampliar o conjunto cego, rotular churn com desfecho real,
testar embeddings (ex.: BERTimbau) e otimizar limiares por custo de negócio.